# Isolation Study -MVSA Single

<a id="top"></a>

  <div style="flex: 1; background-color: #FFEBEE; border: 1px solid #dee2e6; border-left: 10px solid #6c757d; padding: 15px; border-radius: 8px;">
  <strong>Ablation:</strong>
  <ul>
    <li><a href="#iso">Isolating Fusion-Only Speedup</a></li> 
    <li><a href="#concat">Concat</a></li> 
    <li><a href="#noise">Noise robustness</a></li>      
    <li><a href="#fusionrule">Fusion Rule Comparision</a></li>
  </ul>
</div>

In [3]:
import pickle
import numpy as np
import xgboost as xgb
import time
from sklearn.ensemble import ExtraTreesClassifier
import cs16.DPF as DPF #load Private Library CS16
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#load from saved pkl
with open("cache/dpf_mvsa_single_results.pkl", "rb") as f:
    results = pickle.load(f)

# RETRIEVE  Varibles from results
train_probs_text   = results["train_probs_text"]
train_probs_image  = results["train_probs_image"]

val_probs_text     = results["val_probs_text"]
val_probs_image    = results["val_probs_image"]

test_probs_text    = results["test_probs_text"]
test_probs_image   = results["test_probs_image"]

y_train_int = results["y_train_int"]
y_val_int   = results["y_val_int"]
y_test_int  = results["y_test_int"]

beta  = results["beta"]
topk  = results["topk"]
delta = results["delta"]

# DPF  1 test

In [19]:
beta = 1.1
topk=2
delta =1e-2
# Dynamic Fusion
X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(train_probs_text, train_probs_image, beta, topk, delta)
X_val_global   = DPF.enhanced_dynamic_fusion_topk_adaptive(val_probs_text,   val_probs_image, beta, topk, delta)
X_test_global  = DPF.enhanced_dynamic_fusion_topk_adaptive(test_probs_text,  test_probs_image, beta, topk, delta)
print("Dynamic Fusion Feature shape (train):", X_train_global.shape)
# 5. Global Classifier
global_clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
global_clf.fit(X_train_global, y_train_int)

# test
test_pred = global_clf.predict(X_test_global)
print("Test Accuracy:", accuracy_score(y_test_int, test_pred))
print("Test Classification Report:")
print(classification_report(y_test_int, test_pred,
                            digits=4,
                            target_names=['negative','neutral','positive']))

Dynamic Fusion Feature shape (train): (3895, 6)
Test Accuracy: 0.5708418891170431
Test Classification Report:
              precision    recall  f1-score   support

    negative     0.3793    0.1429    0.2075        77
     neutral     0.6350    0.7698    0.6959       278
    positive     0.4380    0.4015    0.4190       132

    accuracy                         0.5708       487
   macro avg     0.4841    0.4381    0.4408       487
weighted avg     0.5412    0.5708    0.5436       487



<a id="iso"></a>
# Isolating Fusion-Only Speedup

[↑ Back to Top](#top)

In [24]:

# ========== 1. MEASURE FUSION TIME ONLY ==========
print("="*50)
print("TIMING: Fusion Only")
print("="*50)

# Run multiple times to get average (e.g., 10 runs)
n_runs = 100
fusion_times = []

for i in range(n_runs):
    start = time.perf_counter()
    
    X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
        train_probs_text, train_probs_image, beta, topk, delta
    )
    X_val_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
        val_probs_text, val_probs_image, beta, topk, delta
    )
    X_test_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
        test_probs_text, test_probs_image, beta, topk, delta
    )
    
    end = time.perf_counter()
    fusion_times.append((end - start) * 1000)  # Convert to milliseconds

print(f"Fusion time (avg over {n_runs} runs): {np.mean(fusion_times):.3f} ± {np.std(fusion_times):.3f} ms")
print(f"X_train_global shape: {X_train_global.shape}")

# ========== 2. XGBOOST TRAINING TIME (OPTIONAL) ==========
print("\n" + "="*50)
print("TIMING: XGBoost Training")
print("="*50)

start = time.perf_counter()
global_clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
global_clf.fit(X_train_global, y_train_int)
train_time = (time.perf_counter() - start) * 1000
print(f"XGBoost training time: {train_time:.3f} ms")

# ========== 3. XGBOOST PREDICTION TIME (OPTIONAL) ==========
print("\n" + "="*50)
print("TIMING: XGBoost Prediction")
print("="*50)

# Run multiple times to get average
pred_times = []
for i in range(n_runs):
    start = time.perf_counter()
    test_pred = global_clf.predict(X_test_global)
    end = time.perf_counter()
    pred_times.append((end - start) * 1000)

print(f"XGBoost prediction time (avg over {n_runs} runs): {np.mean(pred_times):.3f} ± {np.std(pred_times):.3f} ms")

# ========== 4. PERFORMANCE EVALUATION ==========
print("\n" + "="*50)
print("PERFORMANCE")
print("="*50)
test_pred = global_clf.predict(X_test_global)
print(f"Test Accuracy: {accuracy_score(y_test_int, test_pred):.4f}")
print(classification_report(y_test_int, test_pred, digits=4,
                            target_names=['negative', 'neutral', 'positive']))

TIMING: Fusion Only
Fusion time (avg over 100 runs): 1.174 ± 0.154 ms
X_train_global shape: (3895, 6)

TIMING: XGBoost Training
XGBoost training time: 854.006 ms

TIMING: XGBoost Prediction
XGBoost prediction time (avg over 100 runs): 1.902 ± 0.364 ms

PERFORMANCE
Test Accuracy: 0.5708
              precision    recall  f1-score   support

    negative     0.3793    0.1429    0.2075        77
     neutral     0.6350    0.7698    0.6959       278
    positive     0.4380    0.4015    0.4190       132

    accuracy                         0.5708       487
   macro avg     0.4841    0.4381    0.4408       487
weighted avg     0.5412    0.5708    0.5436       487



<a id="concat"></a>
# Concat
[↑ Back to Top](#top)

In [23]:
# ========== CONCAT BASELINE (FULL PIPELINE) ==========
print("="*50)
print("CONCAT BASELINE: Full Pipeline")
print("="*50)

# 1. Fusion time
n_runs_concat = 100
concat_fusion_times = []

for i in range(n_runs_concat):
    start = time.perf_counter()
    X_train_concat = np.concatenate([train_probs_text, train_probs_image], axis=1)
    X_val_concat = np.concatenate([val_probs_text, val_probs_image], axis=1)
    X_test_concat = np.concatenate([test_probs_text, test_probs_image], axis=1)
    end = time.perf_counter()
    concat_fusion_times.append((end - start) * 1000)

print(f"Concat fusion time (avg over {n_runs_concat} runs): {np.mean(concat_fusion_times):.3f} ± {np.std(concat_fusion_times):.3f} ms")

# 2. XGBoost training time
start = time.perf_counter()
concat_clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
concat_clf.fit(X_train_concat, y_train_int)
concat_train_time = (time.perf_counter() - start) * 1000
print(f"Concat XGBoost training time: {concat_train_time:.3f} ms")

# 3. XGBoost prediction time
concat_pred_times = []
for i in range(n_runs_concat):
    start = time.perf_counter()
    concat_pred = concat_clf.predict(X_test_concat)
    end = time.perf_counter()
    concat_pred_times.append((end - start) * 1000)

print(f"Concat XGBoost prediction time (avg over {n_runs_concat} runs): {np.mean(concat_pred_times):.3f} ± {np.std(concat_pred_times):.3f} ms")

# 4. Performance evaluation
print("\n" + "="*50)
print("CONCAT PERFORMANCE")
print("="*50)
print(f"Test Accuracy: {accuracy_score(y_test_int, concat_pred):.4f}")
print(classification_report(y_test_int, concat_pred, digits=4,
                            target_names=['negative', 'neutral', 'positive']))

CONCAT BASELINE: Full Pipeline
Concat fusion time (avg over 100 runs): 0.087 ± 0.089 ms
Concat XGBoost training time: 879.169 ms
Concat XGBoost prediction time (avg over 100 runs): 1.844 ± 0.245 ms

CONCAT PERFORMANCE
Test Accuracy: 0.5606
              precision    recall  f1-score   support

    negative     0.5625    0.1169    0.1935        77
     neutral     0.6021    0.8165    0.6931       278
    positive     0.3936    0.2803    0.3274       132

    accuracy                         0.5606       487
   macro avg     0.5194    0.4046    0.4047       487
weighted avg     0.5393    0.5606    0.5150       487



In [47]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb

# Fixed unimodal outputs
p_text = test_probs_text
p_image = test_probs_image
y_true = y_test_int

# Define fusion rules
def concat_fusion(p_text, p_image):
    return np.concatenate([p_text, p_image], axis=1)

def average_fusion(p_text, p_image):
    return (p_text + p_image) / 2

def weighted_avg_fusion(p_text, p_image, beta=0.5):
    return beta * p_text + (1 - beta) * p_image

def max_fusion(p_text, p_image):
    return np.maximum(p_text, p_image)

def product_fusion(p_text, p_image):
    prod = p_text * p_image
    return prod / np.sum(prod, axis=1, keepdims=True)

def dpf_fusion(p_text, p_image):
    wx = DPF.dynamic_weight_topk_adaptive(p_text, p_image, beta, topk, delta)
    fused_text = wx[:, 0:1] * p_text
    fused_image = wx[:, 1:2] * p_image
    return np.concatenate([fused_text, fused_image], axis=1)

# Train XGBoost for each fusion rule
fusion_rules = {
    "Concat": concat_fusion,
    "Average": average_fusion,
    "Weighted (β=0.5)": lambda p,i: weighted_avg_fusion(p,i,0.5),
    "Max": max_fusion,
    "Product": product_fusion,
    "DPF (Ours)": dpf_fusion,
}

results = {}
for name, fusion_func in fusion_rules.items():
    print(f"\n>>> {name}")
    
    # Fusion
    X_train = fusion_func(train_probs_text, train_probs_image)
    X_test = fusion_func(test_probs_text, test_probs_image)
    
    # Train XGBoost
    clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
    clf.fit(X_train, y_train_int)
    
    # Predict & evaluate
    pred = clf.predict(X_test)
    f1 = f1_score(y_test_int, pred, average='weighted')
    acc = accuracy_score(y_test_int, pred)
    
    results[name] = {'F1': f1, 'Accuracy': acc}
    print(f"  F1: {f1:.4f}, Acc: {acc:.4f}")

# Print summary table
print("\n" + "="*50)
print("FUSION RULE COMPARISON")
print("="*50)
print(f"{'Method':<20} {'F1':<10} {'Accuracy':<10}")
print("-"*40)
for name, metrics in sorted(results.items(), key=lambda x: x[1]['F1'], reverse=True):
    print(f"{name:<20} {metrics['F1']:.4f}     {metrics['Accuracy']:.4f}")


>>> Concat
  F1: 0.5150, Acc: 0.5606

>>> Average
  F1: 0.4487, Acc: 0.4374

>>> Weighted (β=0.5)
  F1: 0.4487, Acc: 0.4374

>>> Max
  F1: 0.4965, Acc: 0.5524

>>> Product
  F1: 0.4319, Acc: 0.4168

>>> DPF (Ours)
  F1: 0.5436, Acc: 0.5708

FUSION RULE COMPARISON
Method               F1         Accuracy  
----------------------------------------
DPF (Ours)           0.5436     0.5708
Concat               0.5150     0.5606
Max                  0.4965     0.5524
Average              0.4487     0.4374
Weighted (β=0.5)     0.4487     0.4374
Product              0.4319     0.4168


In [42]:
#2 import numpy as np
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb

# Fixed unimodal outputs
p_text_train = train_probs_text
p_image_train = train_probs_image
p_text_test = test_probs_text
p_image_test = test_probs_image
y_train = y_train_int
y_test = y_test_int

# Define fusion rules (returning probabilities, not concat features)
def average_fusion_prob(p_text, p_image):
    return (p_text + p_image) / 2

def weighted_avg_fusion_prob(p_text, p_image, beta=0.5):
    return beta * p_text + (1 - beta) * p_image

def max_fusion_prob(p_text, p_image):
    return np.maximum(p_text, p_image)

def product_fusion_prob(p_text, p_image):
    prod = p_text * p_image
    return prod / np.sum(prod, axis=1, keepdims=True)

def dpf_fusion_prob(p_text, p_image):
    wx = DPF.dynamic_weight_topk_adaptive(p_text, p_image, beta, topk, delta)
    return wx[:, 0:1] * p_text + wx[:, 1:2] * p_image

# Define fusion rules (returning concat features for XGBoost)
def concat_fusion_feat(p_text, p_image):
    return np.concatenate([p_text, p_image], axis=1)

def dpf_fusion_feat(p_text, p_image):
    wx = DPF.dynamic_weight_topk_adaptive(p_text, p_image, beta, topk, delta)
    fused_text = wx[:, 0:1] * p_text
    fused_image = wx[:, 1:2] * p_image
    return np.concatenate([fused_text, fused_image], axis=1)

# ========== VERSION 1: ARGMAX (no downstream confounder) ==========
print("="*60)
print("VERSION 1: ARGMAX (no downstream classifier)")
print("="*60)

fusion_rules_prob = {
    "Average": average_fusion_prob,
    "Weighted (β=0.5)": lambda p,i: weighted_avg_fusion_prob(p,i,0.5),
    "Max": max_fusion_prob,
    "Product": product_fusion_prob,
    "DPF (Ours)": dpf_fusion_prob,
}

results_argmax = {}
for name, fusion_func in fusion_rules_prob.items():
    fused_test = fusion_func(p_text_test, p_image_test)
    pred = np.argmax(fused_test, axis=1)
    f1 = f1_score(y_test, pred, average='weighted')
    acc = accuracy_score(y_test, pred)
    results_argmax[name] = {'F1': f1, 'Accuracy': acc}
    print(f"{name:<20} F1: {f1:.4f}, Acc: {acc:.4f}")

# ========== VERSION 2: XGBOOST (downstream classifier) ==========
print("\n" + "="*60)
print("VERSION 2: XGBOOST (downstream classifier)")
print("="*60)

fusion_rules_feat = {
    "Concat": concat_fusion_feat,
    "DPF (Ours)": dpf_fusion_feat,
}

results_xgb = {}
for name, fusion_func in fusion_rules_feat.items():
    X_train = fusion_func(p_text_train, p_image_train)
    X_test = fusion_func(p_text_test, p_image_test)
    
    clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    f1 = f1_score(y_test, pred, average='weighted')
    acc = accuracy_score(y_test, pred)
    results_xgb[name] = {'F1': f1, 'Accuracy': acc}
    print(f"{name:<20} F1: {f1:.4f}, Acc: {acc:.4f}")

# ========== SUMMARY ==========
print("\n" + "="*60)
print("SUMMARY: Argmax vs XGBoost")
print("="*60)
print(f"{'Method':<20} {'Argmax F1':<12} {'XGBoost F1':<12}")
print("-"*50)
print(f"{'Average':<20} {results_argmax['Average']['F1']:.4f}       -")
print(f"{'Weighted (β=0.5)':<20} {results_argmax['Weighted (β=0.5)']['F1']:.4f}       -")
print(f"{'Max':<20} {results_argmax['Max']['F1']:.4f}       -")
print(f"{'Product':<20} {results_argmax['Product']['F1']:.4f}       -")
print(f"{'Concat':<20} -              {results_xgb['Concat']['F1']:.4f}")
print(f"{'DPF (Ours)':<20} {results_argmax['DPF (Ours)']['F1']:.4f}      {results_xgb['DPF (Ours)']['F1']:.4f}")

VERSION 1: ARGMAX (no downstream classifier)
Average              F1: 0.4816, Acc: 0.5770
Weighted (β=0.5)     F1: 0.4816, Acc: 0.5770
Max                  F1: 0.4865, Acc: 0.5729
Product              F1: 0.4841, Acc: 0.5811
DPF (Ours)           F1: 0.4750, Acc: 0.5729

VERSION 2: XGBOOST (downstream classifier)
Concat               F1: 0.5150, Acc: 0.5606
DPF (Ours)           F1: 0.5436, Acc: 0.5708

SUMMARY: Argmax vs XGBoost
Method               Argmax F1    XGBoost F1  
--------------------------------------------------
Average              0.4816       -
Weighted (β=0.5)     0.4816       -
Max                  0.4865       -
Product              0.4841       -
Concat               -              0.5150
DPF (Ours)           0.4750      0.5436


<a id="noise"></a>
# NOISE ROBUSTNESS SWEEP
[↑ Back to Top](#top)

In [33]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score

# ========== 1. TRAIN CLASSIFIERS FIRST ==========
print("Training classifiers on clean data...")

# DPF fusion on CLEAN data
X_train_dpf = dpf_fusion(train_probs_text, train_probs_image)
xgb_clf_dpf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
xgb_clf_dpf.fit(X_train_dpf, y_train_int)
print(f"DPF classifier trained on clean data")

# Uniform fusion on CLEAN data
X_train_uniform = uniform_fusion(train_probs_text, train_probs_image)
xgb_clf_uniform = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
xgb_clf_uniform.fit(X_train_uniform, y_train_int)
print(f"Uniform classifier trained on clean data")

# ========== 2. NOISE SWEEP ==========
def add_noise(probs, noise_level):
    """Add Gaussian noise and re-normalize"""
    noise = np.random.normal(0, noise_level, probs.shape)
    noisy_probs = probs + noise
    noisy_probs = np.maximum(noisy_probs, 0)
    return noisy_probs / np.sum(noisy_probs, axis=1, keepdims=True)

def uniform_fusion(p_text, p_image):
    return (p_text + p_image) / 2

def dpf_fusion(p_text, p_image):
    wx = DPF.dynamic_weight_topk_adaptive(p_text, p_image, beta, topk, delta)
    fused_text = wx[:, 0:1] * p_text
    fused_image = wx[:, 1:2] * p_image
    return np.concatenate([fused_text, fused_image], axis=1)

noise_levels = [0, 0.05, 0.10, 0.15, 0.20]
results = []

print("\n" + "="*50)
print("NOISE ROBUSTNESS SWEEP")
print("="*50)

for noise in noise_levels:
    print(f"\n>>> Noise: {noise*100:.0f}%")
    
    # Add noise to test probabilities
    p_text_noisy = add_noise(test_probs_text, noise)
    p_image_noisy = add_noise(test_probs_image, noise)
    
    # DPF prediction
    X_test_dpf = dpf_fusion(p_text_noisy, p_image_noisy)
    pred_dpf = xgb_clf_dpf.predict(X_test_dpf)
    f1_dpf = f1_score(y_test_int, pred_dpf, average='weighted')
    
    # Uniform prediction
    X_test_uniform = uniform_fusion(p_text_noisy, p_image_noisy)
    pred_uniform = xgb_clf_uniform.predict(X_test_uniform)
    f1_uniform = f1_score(y_test_int, pred_uniform, average='weighted')
    
    results.append({
        'noise': noise,
        'DPF': f1_dpf,
        'Uniform': f1_uniform,
        'advantage': f1_dpf - f1_uniform
    })
    print(f"  DPF: {f1_dpf:.4f}, Uniform: {f1_uniform:.4f}, Advantage: +{f1_dpf - f1_uniform:.4f}")

# ========== 3. SUMMARY TABLE ==========
print("\n" + "="*50)
print("NOISE ROBUSTNESS SUMMARY")
print("="*50)
print(f"{'Noise':<10} {'DPF':<12} {'Uniform':<12} {'Advantage':<12}")
print("-"*50)
for r in results:
    print(f"{r['noise']*100:.0f}%      {r['DPF']:.4f}      {r['Uniform']:.4f}      +{r['advantage']:.4f}")

Training classifiers on clean data...
DPF classifier trained on clean data
Uniform classifier trained on clean data

NOISE ROBUSTNESS SWEEP

>>> Noise: 0%
  DPF: 0.5436, Uniform: 0.4487, Advantage: +0.0949

>>> Noise: 5%
  DPF: 0.5216, Uniform: 0.4659, Advantage: +0.0556

>>> Noise: 10%
  DPF: 0.4673, Uniform: 0.4524, Advantage: +0.0149

>>> Noise: 15%
  DPF: 0.4632, Uniform: 0.4311, Advantage: +0.0321

>>> Noise: 20%
  DPF: 0.4274, Uniform: 0.4146, Advantage: +0.0128

NOISE ROBUSTNESS SUMMARY
Noise      DPF          Uniform      Advantage   
--------------------------------------------------
0%      0.5436      0.4487      +0.0949
5%      0.5216      0.4659      +0.0556
10%      0.4673      0.4524      +0.0149
15%      0.4632      0.4311      +0.0321
20%      0.4274      0.4146      +0.0128


In [38]:

avg_prob = average_fusion_prob(test_probs_text, test_probs_image)
dpf_prob = dpf_fusion_prob(test_probs_text, test_probs_image)

diff = np.abs(dpf_prob - avg_prob).mean()
print(f"Mean absolute difference: {diff:.6f}")


pred_avg = np.argmax(avg_prob, axis=1)
pred_dpf = np.argmax(dpf_prob, axis=1)
same = (pred_avg == pred_dpf).mean()
print(f"Predictions identical: {same:.1%}")

Mean absolute difference: 0.019377
Predictions identical: 99.4%


<a id="fusionrule"></a>
# Fusion Rule Comparison

[↑ Back to Top](#top)

In [46]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb

# Fixed unimodal outputs
p_text = test_probs_text
p_image = test_probs_image
y_true = y_test_int

# ========== ARGMAX VERSION ==========
print("="*50)
print("ARGMAX VERSION (Direct probability fusion)")
print("="*50)

def uniform_fusion_prob(p_text, p_image):
    return (p_text + p_image) / 2

def dpf_fusion_prob(p_text, p_image):
    wx = DPF.dynamic_weight_topk_adaptive(p_text, p_image, beta, topk, delta)
    return wx[:, 0:1] * p_text + wx[:, 1:2] * p_image

# Uniform argmax
pred_uni = np.argmax(uniform_fusion_prob(p_text, p_image), axis=1)
f1_uni_argmax = f1_score(y_test_int, pred_uni, average='weighted')
print(f"Uniform (argmax): F1 = {f1_uni_argmax:.4f}")

# DPF argmax
pred_dpf = np.argmax(dpf_fusion_prob(p_text, p_image), axis=1)
f1_dpf_argmax = f1_score(y_test_int, pred_dpf, average='weighted')
print(f"DPF (argmax):     F1 = {f1_dpf_argmax:.4f}")
print(f"Advantage: +{f1_dpf_argmax - f1_uni_argmax:.4f}")

# ========== XGBOOST VERSION ==========
print("\n" + "="*50)
print("XGBOOST VERSION (Concat features)")
print("="*50)

def concat_fusion(p_text, p_image):
    return np.concatenate([p_text, p_image], axis=1)

def average_fusion(p_text, p_image):
    return (p_text + p_image) / 2

def weighted_avg_fusion(p_text, p_image, beta=0.5):
    return beta * p_text + (1 - beta) * p_image

def max_fusion(p_text, p_image):
    return np.maximum(p_text, p_image)

def product_fusion(p_text, p_image):
    prod = p_text * p_image
    return prod / np.sum(prod, axis=1, keepdims=True)

def dpf_fusion_concat(p_text, p_image):
    wx = DPF.dynamic_weight_topk_adaptive(p_text, p_image, beta, topk, delta)
    fused_text = wx[:, 0:1] * p_text
    fused_image = wx[:, 1:2] * p_image
    return np.concatenate([fused_text, fused_image], axis=1)

fusion_rules = {
    "Concat": concat_fusion,
    "Average": average_fusion,
    "Weighted (β=0.5)": lambda p,i: weighted_avg_fusion(p,i,0.5),
    "Max": max_fusion,
    "Product": product_fusion,
    "DPF (Ours)": dpf_fusion_concat,
}

results = {}
for name, fusion_func in fusion_rules.items():
    print(f"\n>>> {name}")
    
    X_train = fusion_func(train_probs_text, train_probs_image)
    X_test = fusion_func(test_probs_text, test_probs_image)
    
    clf = xgb.XGBClassifier(n_estimators=350, max_depth=5, random_state=42)
    clf.fit(X_train, y_train_int)
    
    pred = clf.predict(X_test)
    f1 = f1_score(y_test_int, pred, average='weighted')
    acc = accuracy_score(y_test_int, pred)
    
    results[name] = {'F1': f1, 'Accuracy': acc}
    print(f"  F1: {f1:.4f}, Acc: {acc:.4f}")

# ========== SUMMARY WITH DIFFERENCE ==========
print("\n" + "="*70)
print("SUMMARY: Fusion Rule Comparison")
print("="*70)
print(f"{'Method':<20} {'F1 (XGBoost)':<15} {'F1 (argmax)':<15} {'Δ (XGB - argmax)':<18}")
print("-"*70)

# For Average (Uniform in argmax)
avg_xgb = results['Average']['F1']
avg_argmax = f1_uni_argmax
avg_diff = avg_xgb - avg_argmax

# For DPF
dpf_xgb = results['DPF (Ours)']['F1']
dpf_argmax = f1_dpf_argmax
dpf_diff = dpf_xgb - dpf_argmax

print(f"{'Average (Uniform)':<20} {avg_xgb:.4f}          {avg_argmax:.4f}          {avg_diff:+.4f}")
print(f"{'DPF (Ours)':<20} {dpf_xgb:.4f}          {dpf_argmax:.4f}          {dpf_diff:+.4f}")

# Other methods (no argmax version)
for name in results:
    if name not in ['Average', 'DPF (Ours)']:
        print(f"{name:<20} {results[name]['F1']:.4f}          {'N/A':<15} {'N/A':<18}")


ARGMAX VERSION (Direct probability fusion)
Uniform (argmax): F1 = 0.4816
DPF (argmax):     F1 = 0.4750
Advantage: +-0.0066

XGBOOST VERSION (Concat features)

>>> Concat
  F1: 0.5150, Acc: 0.5606

>>> Average
  F1: 0.4487, Acc: 0.4374

>>> Weighted (β=0.5)
  F1: 0.4487, Acc: 0.4374

>>> Max
  F1: 0.4965, Acc: 0.5524

>>> Product
  F1: 0.4319, Acc: 0.4168

>>> DPF (Ours)
  F1: 0.5436, Acc: 0.5708

SUMMARY: Fusion Rule Comparison
Method               F1 (XGBoost)    F1 (argmax)     Δ (XGB - argmax)  
----------------------------------------------------------------------
Average (Uniform)    0.4487          0.4816          -0.0329
DPF (Ours)           0.5436          0.4750          +0.0686
Concat               0.5150          N/A             N/A               
Weighted (β=0.5)     0.4487          N/A             N/A               
Max                  0.4965          N/A             N/A               
Product              0.4319          N/A             N/A               
